# Restaurant Sales Analysis Project
### NoteBook 4: Q4_Data_Cleaning

## 1. Import Libraries

In [1]:
import pandas as pd

## 2. Load Data


In [2]:
df4 = pd.read_excel("../../../Data/raw/Q4.xlsx")

## 3. Initial Data Inspection

This section presents an initial assessment of the dataset, including its structure, data types, missing values, and records with zero values in key financial fields to identify potential data quality issues before the cleaning process.

### 3.1 Verify Dataset Structure

In [3]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53993 entries, 0 to 53992
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    20848 non-null  datetime64[ns]
 1   Receipt Number          20848 non-null  object        
 2   Customer                20848 non-null  object        
 3   Invoice                 20848 non-null  object        
 4   Is Refunded             20848 non-null  float64       
 5   Order Lines/Product     53993 non-null  object        
 6   Order Lines/Quantity    53993 non-null  float64       
 7   Order Lines/Unit Price  53993 non-null  float64       
 8   Order Lines/Subtotal    53993 non-null  float64       
 9   Total                   20848 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(4)
memory usage: 4.1+ MB


### 3.2 Inspect Missing Values

In [4]:
df4.isna().sum()

Date                      33145
Receipt Number            33145
Customer                  33145
Invoice                   33145
Is Refunded               33145
Order Lines/Product           0
Order Lines/Quantity          0
Order Lines/Unit Price        0
Order Lines/Subtotal          0
Total                     33145
dtype: int64

### 3.3 Inspect transaction records for zero values in key financial fields.

In [5]:
# Count transaction records containing zero values in key financial fields.
zero_value_records = df4[
    (df4["Order Lines/Quantity"] == 0) |
    (df4["Order Lines/Subtotal"] == 0) |
    (df4["Total"] == 0) |
    (df4["Order Lines/Unit Price"] == 0)
]

print(f"Records requiring review: {len(zero_value_records)}")

Records requiring review: 16


### 3.4 Check for receipt numbers linked to multiple transaction dates.

In [6]:
df4.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(28) 

Receipt Number
طلب 15077-004-24798    2
طلب 15077-004-24895    2
طلب 15432-001-0082     2
طلب 17124-003-25937    2
طلب 13602-001-2349     2
طلب 16930-001-24513    2
طلب 16213-001-26798    2
طلب 14619-001-1793     2
طلب 16545-003-23783    2
طلب 15097-004-1298     2
طلب 14995-001-1283     2
طلب 16181-001-23527    2
طلب 19048-001-38133    2
طلب 15097-004-0600     2
طلب 21164-004-50716    2
طلب 14995-002-1700     2
طلب 15097-004-0823     2
طلب 13536-001-1748     2
طلب 13602-001-2289     2
طلب 17793-001-32832    2
طلب 15077-004-24836    2
طلب 17153-001-25101    2
طلب 15077-004-24862    2
طلب 15077-004-24857    2
طلب 15077-004-24849    2
طلب 17822-002-31435    2
طلب 15035-001-22312    2
طلب 16930-001-24909    2
Name: Date, dtype: int64

## 4. Rename Columns

Column names were renamed to improve readability and simplify the analysis.

In [7]:
df4.rename(columns={
    "Date": "Date",
    "Receipt Number": "Receipt Number",
    "Customer": "Customer",
    "Invoice": "Invoice",
    "Is Refunded": "Is Refunded", 
    "Order Lines/Product": "Product",
    "Order Lines/Quantity": "Quantity",
    "Order Lines/Unit Price": "Unit_Price",
    "Order Lines/Subtotal": "Subtotal",
    "Total": "Total"
}, inplace=True)

## 5. Data Cleaning, Preprocessing, and Data Consistency Checks

This section performs the main data cleaning and preprocessing steps to improve the quality and consistency of the dataset. First, a copy of the original dataset is created to preserve the raw data. Unnecessary columns (`Invoice` and `Is Refunded`) are removed as they are not required for the analysis.

Next, forward filling is applied only to selected columns (`Date`, `Receipt Number`, `Quantity`, `Unit_Price`, and `Subtotal`) because these values are expected to remain consistent within the same transaction. The `Customer` column is intentionally excluded to avoid assigning incorrect customer names to unrelated transactions, while the `Total` column is excluded because it represents the overall transaction amount rather than individual product lines, and propagating its values could introduce inaccurate financial information. Records with a `Subtotal` value of zero were removed during the cleaning process.

A data consistency check is then performed to identify receipt numbers associated with multiple transaction dates. The inspection revealed a small number of duplicate receipt numbers linked to different transaction times. These duplicate cases were resolved by removing the oldest transaction record for each affected receipt number while retaining the most recent one. Finally, the dataset was inspected for fully duplicated records to ensure data integrity. Any exact duplicate records were identified and removed to eliminate redundant transaction entries. A final validation check was then performed to confirm that no duplicate records remained before proceeding to the exploratory data analysis.

In [8]:
# Create a copy of the original dataset to preserve the raw data.
df4_copy = df4.copy()
df4_copy = df4_copy.drop(columns=['Invoice', 'Is Refunded'])

In [9]:
# Propagate transaction-level values to product rows within the same transaction.
columns_to_fill = [
    "Date",
    "Receipt Number",
    "Quantity",
    "Unit_Price",
    "Subtotal",

]


df4_copy[columns_to_fill] = df4_copy[columns_to_fill].ffill()

In [10]:
df4_copy.isna().sum()

Date                  0
Receipt Number        0
Customer          33145
Product               0
Quantity              0
Unit_Price            0
Subtotal              0
Total             33145
dtype: int64

In [11]:
df4_copy = df4_copy[df4_copy['Subtotal'] != 0]

In [12]:
# Function Remove the oldest transaction (based on Date) for a given Receipt Number.
def remove_oldest_receipt_record(df, receipt_number):

    # Check if the receipt number exists
    if receipt_number not in df["Receipt Number"].values:
        print(f"Receipt Number '{receipt_number}' not found.")
        return df

    # Find the oldest date for the receipt
    old_date = df.loc[
        df["Receipt Number"] == receipt_number,
        "Date"
    ].min()

    # Remove all rows belonging to the oldest transaction
    df = df[
        ~(
            (df["Receipt Number"] == receipt_number) &
            (df["Date"] == old_date)
        )
    ]

    return df

In [13]:
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 17822-002-31435")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 15035-001-22312")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 16930-001-24909")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 17153-001-25101")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 17793-001-32832")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 13602-001-2289")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 13536-001-1748")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 14995-002-1700")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 21164-004-50716")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 19048-001-38133")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 16181-001-23527")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 14995-001-1283")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 16545-003-23783")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 14619-001-1793")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 16213-001-26798")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 16930-001-24513")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 13602-001-2349")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 17124-003-25937")
df4_copy = remove_oldest_receipt_record(df4_copy, "طلب 15432-001-0082")

In [14]:
# Check for duplicate records.
print(f"Duplicate records: {df4_copy.duplicated().sum()}")

Duplicate records: 32


In [15]:
# Remove duplicate records.
df4_copy = df4_copy.drop_duplicates()

In [16]:
# Verify that no duplicate records remain.
print(f"Duplicate records after removal: {df4_copy.duplicated().sum()}")

Duplicate records after removal: 0


## 6. Export Clean Dataset

In [17]:
df4_copy.to_excel("../../../Data/Cleaned/clean_q4_data.xlsx", index=False)